# 01 - Acondicionamiento del dataset

## Objetivo

Realizar una inspección inicial del archivo original Chronic Kidney Disease
para identificar su estructura, características y posibles inconsistencias
en los registros antes de comenzar el análisis exploratorio de datos (EDA).

En esta etapa se documentarán las características del archivo, se
verificará la integridad de los registros y se realizarán las correcciones
necesarias para obtener un dataset estructurado y consistente.

In [47]:
import sys
from pathlib import Path
import pandas as pd
import importlib

# 1. Encontrar automáticamente la raíz del proyecto
PROJECT_ROOT = None

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "helpers").is_dir():
        PROJECT_ROOT = folder
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se encontró la carpeta 'helpers'. "
        "Verificá desde qué directorio se está ejecutando la notebook."
    )

# 2. Agregar la raíz del proyecto a las rutas de búsqueda
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Raíz del proyecto: {PROJECT_ROOT}")

# 3. Importar y recargar el helper
import helpers.arff_utils

importlib.reload(helpers.arff_utils)

from helpers.arff_utils import (
    read_arff_header,
    find_inconsistent_rows,
    read_arff_attributes,
    correct_arff_format_line_215_218_515,
    validate_categorical_values,
    validate_numeric_values,
    count_arff_records
)

# 4. Ruta del dataset original
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "chronic_kidney_disease_full.arff"

Raíz del proyecto: c:\Users\Meli\OneDrive\Documentos\Maestría IA\Especialización IA\CEIA_AndD\Chronic-Kidney-Disease-EDA-ActividadIndividual2-Piacentino


## 1. Inspección del archivo original

### 1.1. Encabezado y metadatos

Los archivos ARFF (Attribute-Relation File Format) contienen una sección
de metadatos que describe el conjunto de datos, sus atributos y sus
tipos de variables. Esta información se encuentra antes de la declaración
`@data`, que indica el comienzo de los registros.

A continuación, se presenta el encabezado original del dataset.

In [48]:
header = read_arff_header(DATA_PATH)

print(header)

% 1. Title: Early stage of Indians Chronic Kidney Disease(CKD)
%
% 2. Source Information:
%   (a) Source: 
%			Dr.P.Soundarapandian.M.D.,D.M
%		     (Senior Consultant Nephrologist), 
%			Apollo  Hospitals, 
%			Managiri,
%			Madurai Main Road, 
%			Karaikudi,
%			Tamilnadu,
%			India.
%   (b) Creator: 
%			L.Jerlin Rubini(Research Scholar)
%			Alagappa University
%			EmailId   :jel.jerlin@gmail.com
%			ContactNo :+91-9597231281
%   (c) Guided by: 
%			Dr.P.Eswaran Assistant Professor,
%			Department of Computer Science and Engineering,
%			Alagappa University,
%			Karaikudi,
%			Tamilnadu,
%			India.
%			Emailid:eswaranperumal@gmail.com
%   (d) Date     : july 2015
%
% 3.Relevant Information:
%			age		-	age	
%			bp		-	blood pressure
%			sg		-	specific gravity
%			al		-   	albumin
%			su		-	sugar
%			rbc		-	red blood cells
%			pc		-	pus cell
%			pcc		-	pus cell clumps
%			ba		-	bacteria
%			bgr		-	blood glucose random
%			bu		-	blood urea
%			sc		-	serum creatinine
%			sod		-	sodium
%	

Se extraen del encabezado del archivo ARFF los metadatos de cada atributo, incluyendo su nombre, tipo de dato y, cuando corresponde, los valores permitidos para las variables categóricas.

In [49]:
attributes = read_arff_attributes(DATA_PATH)

print(f"Cantidad de atributos: {len(attributes)}")

for i, attribute in enumerate(attributes):
    print(f"{i + 1:2}. {attribute['name']}")
    print(f"    Tipo: {attribute['type']}")
    print(f"    Valores permitidos: {attribute['allowed_values']}")

Cantidad de atributos: 25
 1. age
    Tipo: numeric
    Valores permitidos: None
 2. bp
    Tipo: numeric
    Valores permitidos: None
 3. sg
    Tipo: nominal
    Valores permitidos: ['1.005', '1.010', '1.015', '1.020', '1.025']
 4. al
    Tipo: nominal
    Valores permitidos: ['0', '1', '2', '3', '4', '5']
 5. su
    Tipo: nominal
    Valores permitidos: ['0', '1', '2', '3', '4', '5']
 6. rbc
    Tipo: nominal
    Valores permitidos: ['normal', 'abnormal']
 7. pc
    Tipo: nominal
    Valores permitidos: ['normal', 'abnormal']
 8. pcc
    Tipo: nominal
    Valores permitidos: ['present', 'notpresent']
 9. ba
    Tipo: nominal
    Valores permitidos: ['present', 'notpresent']
10. bgr
    Tipo: numeric
    Valores permitidos: None
11. bu
    Tipo: numeric
    Valores permitidos: None
12. sc
    Tipo: numeric
    Valores permitidos: None
13. sod
    Tipo: numeric
    Valores permitidos: None
14. pot
    Tipo: numeric
    Valores permitidos: None
15. hemo
    Tipo: numeric
    Valores pe

### 1.2. Verificación de la estructura de los registros

El dataset contiene 24 variables predictoras y una variable objetivo, por lo que cada registro debe contener 25 campos.

Se realiza una inspección de los registros para identificar aquellos que presentan una cantidad diferente de campos respecto de la esperada.

In [50]:
inconsistent_rows = find_inconsistent_rows(
    DATA_PATH,
    expected_fields=25
)

print(f"Cantidad de registros inconsistentes: {len(inconsistent_rows)}")
print()

for row in inconsistent_rows:
    print(f"Línea {row['line_number']}: {row['field_count']} campos")
    print(repr(row["content"]))
    print()

Cantidad de registros inconsistentes: 3

Línea 215: 26 campos
'26,70,1.015,0,4,?,normal,notpresent,notpresent,250,20,1.1,?,?,15.6,52,6900,6.0,no,yes,no,good,no,no,ckd,'

Línea 218: 26 campos
'64,90,1.010,3,3,?,abnormal,present,notpresent,?,35,1.3,?,?,10.3,?,?,?,yes,yes,no,good,yes,no,ckd,'

Línea 515: 26 campos
'75,70,1.020,0,0,normal,normal,notpresent,notpresent,107,48,0.8,144,3.5,13.6,46,10300,4.8,no,,no,no,good,no,no,notckd'



**Líneas 215 y 218:**

En los registros correspondientes a las líneas 215 y 218 se identificó una coma sobrante al final de cada línea, que generaba un campo vacío adicional. Se eliminaron dichas comas para asegurar que cada registro contenga los 25 campos correspondientes a los atributos declarados en el encabezado.

**Línea 515:**

En el registro correspondiente a la línea 515 se identificó una coma adicional entre dos valores consecutivos, que generaba un campo vacío y desplazaba los valores posteriores. Se eliminó la coma redundante para restablecer la correspondencia entre los 25 campos y los atributos declarados en el encabezado.

## 2. Corrección y guardado de dataset

In [51]:
PROCESSED_PATH = (PROJECT_ROOT/ "data"/ "processed"/ "chronic_kidney_disease_fixed.arff")

correct_arff_format_line_215_218_515(input_path=DATA_PATH, output_path=PROCESSED_PATH)

Correcciones aplicadas correctamente.
Registros verificados: todos tienen 25 campos.
Archivo guardado en: c:\Users\Meli\OneDrive\Documentos\Maestría IA\Especialización IA\CEIA_AndD\Chronic-Kidney-Disease-EDA-ActividadIndividual2-Piacentino\data\processed\chronic_kidney_disease_fixed.arff


## 3. Verificación de datos en archivo procesado

### 3.1. Validación de valores categóricos

Se verificó que los valores presentes en el dataset para las variables categóricas pertenezcan a los conjuntos de valores permitidos declarados en el encabezado del archivo ARFF.

Por ejemplo, la variable `rbc` tiene dos categorías posibles: `normal` y `abnormal`, como se observa en la sección `1.1. Encabezado y metadatos` con la función `read_arff_attributes()`. La validación comprueba que los valores registrados para este atributo correspondan a alguna de esas categorías, identificando cualquier valor diferente que pudiera deberse a un error de escritura o de formato.

Para realizar esta comprobación se utilizó la función `validate_categorical_values()`, que compara los valores observados en cada atributo con sus categorías declaradas. Los valores faltantes representados por `?` se excluyeron de la validación.

In [52]:
categorical_issues = validate_categorical_values(PROCESSED_PATH)

print(
    f"Cantidad de valores categóricos no reconocidos: "
    f"{len(categorical_issues)}"
)

Cantidad de valores categóricos no reconocidos: 0


**Resultado:** no se encontraron valores categóricos no reconocidos. Todos los valores observados coinciden con las categorías permitidas en el encabezado.

### 3.2. Validación de valores numéricos

Se verificó que los valores de las variables numéricas puedan interpretarse correctamente como números, de acuerdo con los tipos de datos declarados en el encabezado del archivo ARFF.

Por ejemplo, la variable `age` (edad del paciente) está declarada como `numeric`. La validación comprueba que sus valores puedan convertirse a números, identificando posibles campos vacíos, valores no numéricos o valores no finitos, como `NaN`, `inf` o `-inf`.

Para realizar esta comprobación se utilizó la función `validate_numeric_values()`. Los datos faltantes representados mediante `?` se consideran válidos y se excluyen de esta comprobación.

En esta etapa no se evalúan rangos ni límites clínicos de las variables, sino únicamente la consistencia de sus formatos y tipos de datos.

In [53]:
numeric_issues = validate_numeric_values(PROCESSED_PATH)

print(
    f"Cantidad de inconsistencias en variables numéricas: "
    f"{len(numeric_issues)}"
)

Cantidad de inconsistencias en variables numéricas: 0


**Resultado:** no se encontraron inconsistencias en las variables numéricas. Todos los valores registrados pueden interpretarse correctamente como números y no se identificaron campos vacíos, valores no numéricos ni valores no finitos. 

Los valores faltantes representados mediante `?` fueron excluidos de esta validación.

### 3.3. Verificación de la cantidad de registros

Se contabilizaron los registros del archivo ARFF corregido, excluyendo líneas vacías y comentarios. También se verificó la cantidad de campos presentes en cada registro.

In [55]:
results = count_arff_records(PROCESSED_PATH)

print(f"Cantidad de registros: {results['record_count']}")
print(f"Cantidad de atributos por registro: {results['field_counts']}")

Cantidad de registros: 400
Cantidad de atributos por registro: {25}


**Resultado:** el archivo contiene 400 registros, cada uno con 25 campos, en concordancia con la estructura declarada en el encabezado del dataset.